In [ ]:
from _common import *

## Notebook 04 -- XGBoost

### Section 1 -- MLflow tracking, imports, budget constants

In [ ]:
import mlflow

MLFLOW_DB_PATH = ROOT_FOLDER / 'notebooks' / 'mlflow_xgboost.db'
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")
mlflow.set_experiment("04_xgboost")

In [ ]:
import optuna
from optuna.samplers import TPESampler
from xgboost import XGBRegressor
import json

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
MAX_TRIALS = 100
PATIENCE = 15

### Section 2 -- make_model() and the search space

In [ ]:
import time


def make_model(params):
    return XGBRegressor(
        **params,
        random_state=RANDOM_SEED,
        device='cuda',
        tree_method='hist',
        n_jobs=-1,
    )


def make_objective(fold):
    def objective(trial):
        t0 = time.time()
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 1000, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        }
        score = inner_cv_score(fold, params, make_model)
        print(f"  trial {trial.number}: {time.time() - t0:.1f}s, n_estimators={params['n_estimators']}, score={score:.3f}")
        return score
    return objective


def stop_when_flat(study, trial):
    # study-level patience -- the only stopping rule (Optuna pruning is OFF, see
    # plan doc traps). Stop once PATIENCE trials have passed with no new best.
    if trial.number - study.best_trial.number >= PATIENCE:
        study.stop()

### Rung 1 -- nested tuning, all session folds

In [ ]:
def _json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj


def _save_checkpoint(results):
    with open(FOLD_METRICS_PATH, 'w') as f:
        json.dump(_json_safe(results), f, indent=2)


FOLD_METRICS_PATH = ROOT_FOLDER / 'notebooks' / 'fold_metrics_xgboost.json'

if FOLD_METRICS_PATH.exists():
    with open(FOLD_METRICS_PATH) as f:
        fold_results = json.load(f)
else:
    fold_results = []

done_keys = {(r['participant_id'], r['test_session']) for r in fold_results}
print(f"resuming with {len(fold_results)} folds already done, {len(SESSION_FOLDS) - len(fold_results)} remaining")

In [ ]:
study_first_new = None   # the first fold actually run THIS session -- used for the
                          # convergence diagnostic below, not for scoring

for fold in SESSION_FOLDS:
    p_num = fold['participant_id']
    test_session = fold['test_session']

    if (p_num, test_session) in done_keys:
        continue

    sampler = TPESampler(seed=RANDOM_SEED)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(make_objective(fold), n_trials=MAX_TRIALS, callbacks=[stop_when_flat])

    if study_first_new is None:
        study_first_new = study

    best_params = dict(study.best_params)

    train_mask = (FEATURES['participant_id'] == p_num) & (FEATURES['session_id'].isin(fold['train_sessions']))
    test_mask = (FEATURES['participant_id'] == p_num) & (FEATURES['session_id'] == test_session)

    # leak guard -- the outer test session must never appear in the training rows
    assert not (FEATURES.loc[train_mask, 'session_id'] == test_session).any()
    assert set(FEATURES.loc[train_mask, 'session_id'].unique()) == set(fold['train_sessions'])

    X_train = FEATURES.loc[train_mask, FEATURE_COLS]
    y_train = FEATURES.loc[train_mask, 'pct_mvc_corr']
    X_test = FEATURES.loc[test_mask, FEATURE_COLS]
    y_test = FEATURES.loc[test_mask, 'pct_mvc_corr']

    final_model = make_model(best_params)
    final_model.fit(X_train, y_train)
    y_pred = final_model.predict(X_test)

    result = evaluate(y_test, y_pred, FEATURES.loc[test_mask, 'phase'], FEATURES.loc[test_mask, 'target_pct'])

    with mlflow.start_run(run_name=f"p{p_num}_s{test_session}"):
        mlflow.log_params(best_params)
        mlflow.log_metric('rmse', result['overall']['rmse'])
        mlflow.log_metric('mae', result['overall']['mae'])
        mlflow.log_metric('r2', result['overall']['r2'])
        mlflow.log_metric('n_trials', len(study.trials))

    fold_results.append({
        'participant_id': p_num,
        'fold': fold['fold'],
        'test_session': test_session,
        'best_params': best_params,
        'n_trials': len(study.trials),
        'metrics': result,
    })
    done_keys.add((p_num, test_session))
    _save_checkpoint(fold_results)

    print(p_num, test_session, len(study.trials), result['overall'])

print(f"done -- {len(fold_results)} / {len(SESSION_FOLDS)} folds in fold_metrics_xgboost.json")

### Convergence check -- one fold only

In [ ]:
from optuna.visualization import plot_optimization_history

if study_first_new is not None:
    plot_optimization_history(study_first_new)
else:
    print("nothing new ran this session -- no study to plot")

### Rung 2 -- fine-tuned cross-mount (pending)

In [ ]:
# TODO(rung 2): pending slice-size decision. Placeholder -- not run.

### Rung 3 -- cross-subject, cold

Held-out unit is a whole **participant**, not a session -- every session they have is
excluded from training, and there is no fine-tuning (that is rung 4). Uses
`SUBJECT_FOLDS` (8 folds, one per participant) and `INNER_VAL_PARTICIPANTS`
(`_common.py` SS2b) -- the same fixed 3-participant round-robin used by
`03_ridge.ipynb`'s rung 3. Tuning goes through `inner_cv_score_subject()`
(`_common.py` SS6b) instead of `inner_cv_score()`.

Same search space, same `MAX_TRIALS`/`PATIENCE`, same `make_model()` as rung 1 --
only the fold source and the inner-CV function differ. Writes
`fold_metrics_xgboost_rung3.json` (separate checkpoint file, keyed by
`participant_id` not `(participant_id, test_session)`). Logs to the same
`mlflow_xgboost.db` / `04_xgboost` experiment as rung 1, with a distinct run-name
scheme (`rung3_p{p}` vs rung 1's `p{p}_s{s}`) so the two rungs' runs never collide.

**Per the same gate rung 1 used (plan doc SS4c) and Ridge's rung 3 used (SS4d):
time fold 0 before committing to a budget.** Rung 3's training set per fold is
~7-8x rung 1's (7 whole participants' full data vs ~8 sessions of 1 participant),
and rung 1's fit cost already scales with `n_estimators` almost linearly (SS4c) --
do not assume `MAX_TRIALS=100`/`PATIENCE=15` survives unchanged. The known
CPU/GPU device-mismatch prediction warning (SS4c) still applies here, unfixed.

In [ ]:
def make_objective_subject(fold):
    def objective(trial):
        t0 = time.time()
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 1000, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        }
        score = inner_cv_score_subject(fold, INNER_VAL_PARTICIPANTS[fold['fold']], params, make_model)
        print(f"  trial {trial.number}: {time.time() - t0:.1f}s, n_estimators={params['n_estimators']}, score={score:.3f}")
        return score
    return objective

# make_model() and stop_when_flat() are reused unchanged from rung 1 (cell above).

#### Fold-0 timing check -- confirm the budget before the full 8-fold run

5 trials, no patience callback -- same smoke-test shape used for rung 1 (SS4c) and
for Ridge's rung 3 (SS4d). Not a real tuning run; purely a per-trial wall-clock
measurement to extrapolate from before spending real compute on all 8 folds.

In [ ]:
_t0 = time.time()
timing_sampler = TPESampler(seed=RANDOM_SEED)
timing_study = optuna.create_study(direction='minimize', sampler=timing_sampler)
timing_study.optimize(make_objective_subject(SUBJECT_FOLDS[0]), n_trials=5)
_t1 = time.time()

per_trial = (_t1 - _t0) / 5
projected_fold_minutes = per_trial * MAX_TRIALS / 60
projected_total_hours = per_trial * MAX_TRIALS * len(SUBJECT_FOLDS) / 3600

print(f"rung 3 fold 0, 5 trials, no patience: {per_trial:.2f} s/trial")
print(f"worst case (no patience firing) at MAX_TRIALS={MAX_TRIALS}: "
      f"{projected_fold_minutes:.1f} min/fold, {projected_total_hours:.1f} h for all {len(SUBJECT_FOLDS)} folds")

#### Rung 3 -- nested tuning, all 8 folds

**Review the timing check above before running this cell.** If the projected total
does not survive (same failure mode as rung 1, plan doc SS4c), lower
`MAX_TRIALS`/`PATIENCE` for rung 3 specifically rather than assuming rung 1's
numbers transfer -- do not silently shrink the search space to compensate without
flagging it.

**Resumable**, same pattern as rung 1: loads `fold_metrics_xgboost_rung3.json` if
present, skips any `participant_id` already in it, saves after every new fold.

In [ ]:
FOLD_METRICS_RUNG3_PATH = ROOT_FOLDER / 'notebooks' / 'fold_metrics_xgboost_rung3.json'

if FOLD_METRICS_RUNG3_PATH.exists():
    with open(FOLD_METRICS_RUNG3_PATH) as f:
        fold_results_rung3 = json.load(f)
else:
    fold_results_rung3 = []

done_keys_rung3 = {r['participant_id'] for r in fold_results_rung3}
print(f"resuming with {len(fold_results_rung3)} folds already done, {len(SUBJECT_FOLDS) - len(fold_results_rung3)} remaining")


def _save_checkpoint_rung3(results):
    with open(FOLD_METRICS_RUNG3_PATH, 'w') as f:
        json.dump(_json_safe(results), f, indent=2)


for fold in SUBJECT_FOLDS:
    test_p = int(fold['test_participant'].removeprefix('participant'))

    if test_p in done_keys_rung3:
        continue

    sampler = TPESampler(seed=RANDOM_SEED)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(make_objective_subject(fold), n_trials=MAX_TRIALS, callbacks=[stop_when_flat])

    best_params = dict(study.best_params)

    train_participant_nums = [int(p.removeprefix('participant')) for p in fold['train_participants']]
    train_mask = FEATURES['participant_id'].isin(train_participant_nums)
    test_mask = FEATURES['participant_id'] == test_p

    # leak guard -- the outer test participant must never appear in the training rows
    assert not (FEATURES.loc[train_mask, 'participant_id'] == test_p).any()
    assert set(FEATURES.loc[train_mask, 'participant_id'].unique()) == set(train_participant_nums)

    X_train = FEATURES.loc[train_mask, FEATURE_COLS]
    y_train = FEATURES.loc[train_mask, 'pct_mvc_corr']
    X_test = FEATURES.loc[test_mask, FEATURE_COLS]
    y_test = FEATURES.loc[test_mask, 'pct_mvc_corr']

    final_model = make_model(best_params)
    final_model.fit(X_train, y_train)
    y_pred = final_model.predict(X_test)

    result = evaluate(y_test, y_pred, FEATURES.loc[test_mask, 'phase'], FEATURES.loc[test_mask, 'target_pct'])

    with mlflow.start_run(run_name=f"rung3_p{test_p}"):
        mlflow.log_params(best_params)
        mlflow.log_metric('rmse', result['overall']['rmse'])
        mlflow.log_metric('mae', result['overall']['mae'])
        mlflow.log_metric('r2', result['overall']['r2'])
        mlflow.log_metric('n_trials', len(study.trials))

    fold_results_rung3.append({
        'participant_id': test_p,
        'fold': fold['fold'],
        'best_params': best_params,
        'n_trials': len(study.trials),
        'metrics': result,
    })
    done_keys_rung3.add(test_p)
    _save_checkpoint_rung3(fold_results_rung3)

    print(test_p, len(study.trials), result['overall'])

print(f"done -- {len(fold_results_rung3)} / {len(SUBJECT_FOLDS)} folds in fold_metrics_xgboost_rung3.json")

In [ ]:
rmses_rung3 = np.array([r['metrics']['overall']['rmse'] for r in fold_results_rung3])

print(f"XGBoost, rung 3, {len(rmses_rung3)} folds")
print(f"mean RMSE {rmses_rung3.mean():.2f} %MVC (SD {rmses_rung3.std(ddof=1):.2f}, "
      f"range {rmses_rung3.min():.2f}-{rmses_rung3.max():.2f})")

# Flat per-rung summary, matching rung 1's style above -- superseded once the
# EchoForce-style per-model table (decision 5) exists.

### Summary

In [ ]:
rmses = [r['metrics']['overall']['rmse'] for r in fold_results]
rmses = np.array(rmses)

print(f"XGBoost, rung 1, {len(rmses)} folds")
print(f"mean RMSE {rmses.mean():.2f} %MVC (SD {rmses.std(ddof=1):.2f}, range {rmses.min():.2f}-{rmses.max():.2f})")